In [2]:
import pandas as pd
df = pd.read_csv('df_twfe.csv',sep=',')
df.head()

,hotel_id,date,region,treated,revenue,price,availability
0,134007,2025-01-01,B,0,111.804644,92.718519,0.514214
1,134007,2025-01-02,B,0,107.839755,92.718519,0.665474
2,134007,2025-01-03,B,0,118.874988,92.718519,0.588169
3,134007,2025-01-04,B,0,117.652584,92.718519,0.574315
4,134007,2025-01-05,B,0,122.347286,92.718519,0.651727


In [3]:
import statsmodels.formula.api as smf

# OLS с контролем переменных
ols_with = smf.ols('revenue ~ treated + region + price + availability', data=df).fit()
ate_with = ols_with.params['treated']
print('OLS с контролем:', round(ate_with, 1))

# OLS без контроля переменных
ols_without = smf.ols('revenue ~ treated', data=df).fit()
ate_without = ols_without.params['treated']
print('OLS без контроля:', round(ate_without, 1))

# Разница
diff = ate_with - ate_without
print('Разница:', round(diff, 1))

OLS с контролем: 12.4
OLS без контроля: 19.2
Разница: -6.8


In [4]:
# Шаг 1: вычитаем среднее по каждому объекту (hotel_id)
df['availability_mean_hotel'] = df.groupby('hotel_id')['availability'].transform('mean')

# Шаг 2: вычитаем среднее по каждому моменту времени (date)
df['availability_mean_date'] = df.groupby('date')['availability'].transform('mean')

# Шаг 3: demeaning = Yit - Yi - Yt
df['availability_demeaned'] = df['availability'] - df['availability_mean_hotel'] - df['availability_mean_date']

# Среднее преобразованной колонки
print(round(df['availability_demeaned'].mean(), 1))

-0.5


In [5]:
# Шаг 1: demeaning для всех переменных
for col in ['revenue', 'treated', 'availability', 'price']:
    df[f'{col}_mean_hotel'] = df.groupby('hotel_id')[col].transform('mean')
    df[f'{col}_mean_date'] = df.groupby('date')[col].transform('mean')
    df[f'{col}_dm'] = df[col] - df[f'{col}_mean_hotel'] - df[f'{col}_mean_date']

# Шаг 2: OLS на преобразованных переменных с кластеризованными ошибками
twfe_dm = smf.ols('revenue_dm ~ treated_dm + availability_dm + price_dm', data=df).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['hotel_id']}
)
ate_dm = twfe_dm.params['treated_dm']
print('TWFE через demeaning:', round(ate_dm, 1))

# Шаг 3: проверка через dummy-переменные
twfe_dummy = smf.ols('revenue ~ treated + availability + price + C(hotel_id) + C(date)', data=df).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['hotel_id']}
)
ate_dummy = twfe_dummy.params['treated']
print('TWFE через dummy:', round(ate_dummy, 1))

TWFE через demeaning: 15.0
TWFE через dummy: 15.0
